# Question 1

Import data from CSV

In [32]:
import pandas as pd
import numpy as np
import csv

raw_data = pd.read_csv('./data.tsv', sep='\t', quoting=csv.QUOTE_NONE)

Prepare Data, by sampling and adding the label

In [33]:
def label(rating):
    if rating > 3:
        return  1
    if rating < 3:
        return 2
    if rating == 3:
        return 3

sampled = raw_data.groupby("star_rating").sample(n=50000, random_state=42)
del raw_data

dataset = pd.DataFrame()
dataset["review"] = sampled["review_body"]
dataset["star_rating"] = sampled["star_rating"]
dataset["sentiment"] = dataset["star_rating"].apply(label)

del sampled


We will perform train / test split after extracting the features from the sentences

# Question 2(a)

Import gensim and load pre-trained model

In [34]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')

Test word embeddings and semantics

In [5]:
# Example comparing king, man and woman. Expecting to see queen
wx = wv['king'] - wv['man'] + wv['woman']
wv.most_similar(wx, topn=5)

[('king', 0.8449392318725586),
 ('queen', 0.7300517559051514),
 ('monarch', 0.645466148853302),
 ('princess', 0.6156251430511475),
 ('crown_prince', 0.5818676352500916)]

In [6]:
# Example comparing boy, man and puppy. Expecting to see dog
wx = wv['man'] - wv['boy'] + wv['puppy']
wv.most_similar(wx, topn=5)

[('puppy', 0.7963186502456665),
 ('dog', 0.730284571647644),
 ('pooch', 0.6746339201927185),
 ('puppies', 0.6635603904724121),
 ('cat', 0.659332275390625)]

# Question 2(b)

Create model from reviews

In [4]:
from gensim import utils
import gensim.models

class Corpus:
    def __iter__(self):
        for sentence in dataset["review"].tolist():
            yield utils.simple_preprocess(str(sentence))

sentences = Corpus()
model = gensim.models.Word2Vec(sentences=sentences, vector_size=300, window=11)

Retry examples from previous part

In [8]:
wx = model.wv["king"] - model.wv["man"] + model.wv["woman"]
model.wv.most_similar(wx, topn=5)

[('moleskine', 0.40247777104377747),
 ('hybrid', 0.3994394838809967),
 ('woman', 0.3952346742153168),
 ('notebooks', 0.38341987133026123),
 ('softer', 0.3777081370353699)]

In [9]:
wx = model.wv['man'] - model.wv['boy'] + model.wv['puppy']
model.wv.most_similar(wx, topn=5)

[('man', 0.6377042531967163),
 ('woman', 0.46411392092704773),
 ('lady', 0.4263545274734497),
 ('insecure', 0.42187026143074036),
 ('guy', 0.42004823684692383)]

# Question 3

Preprocess sentences and create embeddings

In [5]:
import re
from bs4 import BeautifulSoup
import contractions
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def preprocess_and_vectorize(wv):
    def f(review):
        text = str(review).lower()
        text = BeautifulSoup(text, "html.parser").get_text(strip=True)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = contractions.fix(text)
        stop = set(stopwords.words('english'))
        words = [lemmatizer.lemmatize(w) for w in word_tokenize(text) if w not in stop]
        vectors = [wv[w] if w in wv else [0]*300 for w in words]

        if(len(vectors) == 0): return [0]*300
        else: return np.mean(np.array(vectors), axis=0)
    
    return f;

dataset["feature_pretrained"] = dataset["review"].apply(preprocess_and_vectorize(wv))
dataset["feature_custom"] = dataset["review"].apply(preprocess_and_vectorize(model.wv))

/tmp/ipykernel_59845/3475461715.py:13: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  text = BeautifulSoup(text, "html.parser").get_text(strip=True)
/tmp/ipykernel_59845/3475461715.py:13: MarkupResemblesLocatorWarning: 

In [6]:
del model
del wv

Train Test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(dataset[["feature_pretrained", "feature_custom"]], dataset["sentiment"], test_size=0.2, random_state=42)

General testing skeleton

In [13]:
from sklearn.metrics import accuracy_score

prep_input = lambda x: pd.DataFrame(x.tolist(), index=x.index)

def test_model(model, feature, name):
    model.fit(prep_input(X_train[feature]), y_train)
    y_pred = model.predict(prep_input(X_test[feature]))
    print(f"{name} accuracy: {accuracy_score(y_test, y_pred)}")

In [14]:
from sklearn.linear_model import Perceptron
test_model(Perceptron(random_state=42), "feature_pretrained", "Perceptron with 'word2vec-google-news-300' features")
test_model(Perceptron(random_state=42), "feature_custom", "Perceptron with self trained Word2Vec features")

Perceptron with 'word2vec-google-news-300' features accuracy: 0.53832
Perceptron with self trained Word2Vec features accuracy: 0.60838


In [16]:
from sklearn.svm import LinearSVC
test_model(LinearSVC(random_state=42), "feature_pretrained", "SVM with 'word2vec-google-news-300' features")
test_model(LinearSVC(random_state=42), "feature_custom", "SVM with self trained Word2Vec features")

SVM with 'word2vec-google-news-300' features accuracy: 0.65288
SVM with self trained Word2Vec features accuracy: 0.67464


# Question 4(a)

Prepare inputs

In [ ]:
binary_ds = dataset[dataset["sentiment"] != 3]
binary_ds["sentiment"] = binary_ds["sentiment"].apply(lambda x: 1 if x==1 else 0)
X_train, X_test, y_train, y_test = train_test_split(binary_ds[["feature_pretrained", "feature_custom"]], binary_ds["sentiment"], test_size=0.2, random_state=42)

In [14]:
import keras
from keras.models import Sequential
from keras.layers import Dense

initializer = keras.initializers.HeNormal(seed=42)

model = Sequential([
    Dense(50, input_shape=(300,), kernel_initializer=initializer, activation='relu'),
    Dense(10, activation='relu', kernel_initializer=initializer),
    Dense(2, activation='softmax', kernel_initializer=initializer)
])

model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(prep_input(X_train['feature_pretrained']), pd.get_dummies(y_train), epochs=10, batch_size=50)

Epoch 1/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8092 - loss: 0.4215
Epoch 2/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8292 - loss: 0.3839
Epoch 3/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8354 - loss: 0.3717
Epoch 4/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8403 - loss: 0.3626
Epoch 5/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8445 - loss: 0.3552
Epoch 6/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8472 - loss: 0.3491
Epoch 7/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8489 - loss: 0.3443
Epoch 8/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8517 - loss: 0.3398
Epoch 9/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8533 - loss: 0.3362
Epoch 10/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8552 - loss: 0.3329


In [ ]:
score = model.evaluate(prep_input(X_test['feature_pretrained']), pd.get_dummies(y_test))
print(f"FFN binary classification accuracy with pre-trained embedding: {score[1]}")

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 990us/step - accuracy: 0.8412 - loss: 0.3639
FFN accuracy with pre-trained embedding: 0.8412250280380249


In [17]:
model.fit(prep_input(X_train['feature_custom']), pd.get_dummies(y_train), epochs=10, batch_size=50)

Epoch 1/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8273 - loss: 0.4019
Epoch 2/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8459 - loss: 0.3567
Epoch 3/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8508 - loss: 0.3459
Epoch 4/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8541 - loss: 0.3371
Epoch 5/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8572 - loss: 0.3300
Epoch 6/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8603 - loss: 0.3247
Epoch 7/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8624 - loss: 0.3201
Epoch 8/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8634 - loss: 0.3163
Epoch 9/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8656 - loss: 0.3126
Epoch 10/10
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8679 - loss: 0.3090


In [ ]:
score = model.evaluate(prep_input(X_test['feature_custom']), pd.get_dummies(y_test))
print(f"FFN binary classification accuracy with pre-trained embedding: {score[1]}")

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 939us/step - accuracy: 0.8563 - loss: 0.3358
FFN accuracy with pre-trained embedding: 0.8562999963760376


In [20]:
X_train, X_test, y_train, y_test = train_test_split(dataset[["feature_pretrained", "feature_custom"]], dataset["sentiment"], test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    Dense(50, input_shape=(300,), kernel_initializer=initializer, activation='relu'),
    Dense(10, activation='relu', kernel_initializer=initializer),
    Dense(3, activation='softmax', kernel_initializer=initializer)
])

model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(prep_input(X_train['feature_pretrained']), pd.get_dummies(y_train), epochs=10, batch_size=50)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10


2026-02-27 05:57:12.592678: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 240000000 exceeds 10% of free system memory.


4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6566 - loss: 0.7961
Epoch 2/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6703 - loss: 0.7680
Epoch 3/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6752 - loss: 0.7578
Epoch 4/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6775 - loss: 0.7510
Epoch 5/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6801 - loss: 0.7464
Epoch 6/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6815 - loss: 0.7419
Epoch 7/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6833 - loss: 0.7385
Epoch 8/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6845 - loss: 0.7356
Epoch 9/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6870 - loss: 0.7328
Epoch 10/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6873 - loss: 0.7297


In [25]:
score = model.evaluate(prep_input(X_test['feature_pretrained']), pd.get_dummies(y_test))
print(f"FFN ternary classification accuracy with pre-trained embedding: {score[1]}")

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 2s 976us/step - accuracy: 0.6789 - loss: 0.7614
FFN ternary classification accuracy with pre-trained embedding: 0.6789399981498718


In [28]:
model.fit(prep_input(X_train['feature_custom']), pd.get_dummies(y_train), epochs=10, batch_size=50)

Epoch 1/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.4004 - loss: 1.0551
Epoch 2/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3989 - loss: 1.0551
Epoch 3/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.4003 - loss: 1.0551
Epoch 4/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.4006 - loss: 1.0551
Epoch 5/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3997 - loss: 1.0551
Epoch 6/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3998 - loss: 1.0551
Epoch 7/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3995 - loss: 1.0551
Epoch 8/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.4004 - loss: 1.0551
Epoch 9/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.4011 - loss: 1.0551
Epoch 10/10
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.4012 - loss: 1.0551


In [29]:
score = model.evaluate(prep_input(X_test['feature_pretrained']), pd.get_dummies(y_test))
print(f"FFN ternary classification accuracy with self trained embedding: {score[1]}")

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 2s 974us/step - accuracy: 0.4719 - loss: 1.0159
FFN ternary classification accuracy with self trained embedding: 0.4719200134277344


# Question 4(b)